In [1]:
!pip install pymupdf tqdm pandas unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive

drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

### Configuración de las rutas

In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/UniversidadLLM"

INPUT_DIR = f"{BASE_DIR}/documentos"

RAW_DIR = f"{BASE_DIR}/textoExtraido"

CLEAN_DIR = f"{BASE_DIR}/textoLimpio"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)


### Extracción masiva de PDFs

FUENTE 1
PDFs institucionales
Este script recorre todas las carpetas.

In [ ]:
import fitz
import os
from tqdm import tqdm

def extract_pdf_text(pdf_path):

    text = ""

    try:

        doc = fitz.open(pdf_path)

        for page in doc:

            text += page.get_text("text") + "\n"

        doc.close()

    except Exception as e:

        print(f"Error en {pdf_path}: {e}")

    return text


for root, dirs, files in os.walk(INPUT_DIR):

    for file in tqdm(files):

        if file.lower().endswith(".pdf"):

            pdf_path = os.path.join(root, file)

            relative_path = os.path.relpath(root, INPUT_DIR)

            output_folder = os.path.join(RAW_DIR, relative_path)

            os.makedirs(output_folder, exist_ok=True)

            txt_name = file.replace(".pdf", ".txt")

            txt_path = os.path.join(output_folder, txt_name)

            text = extract_pdf_text(pdf_path)

            with open(txt_path, "w", encoding="utf-8") as f:

                f.write(text)

print("Extracción completada.")

0it [00:00, ?it/s]
100%|██████████| 2/2 [00:00<00:00, 23497.50it/s]

Extracción completada.


Script avanzado de limpieza
### Nueva sección
Este será tu punto de partida

In [ ]:
import re
from unidecode import unidecode

def clean_text(text):

    # eliminar retornos raros
    text = text.replace("\x00", " ")

    # eliminar espacios múltiples
    text = re.sub(r'[ \t]+', ' ', text)

    # eliminar saltos excesivos
    text = re.sub(r'\n{3,}', '\n\n', text)

    # eliminar páginas tipo "Página 4"
    text = re.sub(
        r'Página\s+\d+',
        '',
        text,
        flags=re.IGNORECASE
    )

    # eliminar números de página aislados
    text = re.sub(
        r'\n\s*\d+\s*\n',
        '\n',
        text
    )

    # eliminar URLs
    text = re.sub(
        r'https?://\S+',
        '',
        text
    )

    # eliminar correos
    text = re.sub(
        r'\S+@\S+',
        '',
        text
    )

    text = text.strip()

    return text

### Detectar encabezados repetidos

Los documentos institucionales suelen repetir en cada página.

Lo eliminamos automáticamente.

In [ ]:
#Detectar líneas repetidas

from collections import Counter

def detect_repeated_lines(text):

    lines = [
        line.strip()
        for line in text.splitlines()
        if len(line.strip()) > 5
    ]

    counter = Counter(lines)

    repeated = {
        line
        for line, count in counter.items()
        if count >= 5
    }

    return repeated

In [ ]:
# Se eliminan
def remove_repeated_headers(text):

    repeated = detect_repeated_lines(text)

    cleaned_lines = []

    for line in text.splitlines():

        if line.strip() not in repeated:

            cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

In [ ]:
# Limpieza masiva

for root, dirs, files in os.walk(RAW_DIR):

    for file in tqdm(files):

        if file.endswith(".txt"):

            path = os.path.join(root, file)

            with open(path, "r", encoding="utf-8") as f:

                text = f.read()

            text = remove_repeated_headers(text)

            text = clean_text(text)

            relative_path = os.path.relpath(root, RAW_DIR)

            output_folder = os.path.join(
                CLEAN_DIR,
                relative_path
            )

            os.makedirs(output_folder, exist_ok=True)

            output_path = os.path.join(
                output_folder,
                file
            )

            with open(
                output_path,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(text)

print("Limpieza completada.")

0it [00:00, ?it/s]
100%|██████████| 1/1 [00:00<00:00, 71.41it/s]

Limpieza completada.


Dividir documentos largos en fragmentos
### Esto será útil para generar QA posteriormente.

In [ ]:
def split_text(text, max_chars=2500):

    chunks = []

    start = 0

    while start < len(text):

        end = start + max_chars

        chunk = text[start:end]

        chunks.append(chunk)

        start = end

    return chunks

### Convertimos a JSON

Formato recomendado:

In [ ]:
import json

dataset = []

for root, dirs, files in os.walk(CLEAN_DIR):

    for file in files:

        if file.endswith(".txt"):

            path = os.path.join(root, file)

            categoria = os.path.relpath(
                root,
                CLEAN_DIR
            ).split(os.sep)[0]

            with open(
                path,
                "r",
                encoding="utf-8"
            ) as f:

                text = f.read()

            dataset.append(
                {
                    "categoria": categoria,
                    "documento": file,
                    "texto": text
                }
            )

with open(
    f"{BASE_DIR}/dataSet/corpus.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dataset,
        f,
        ensure_ascii=False,
        indent=2
    )

Validación de calidad
### Nueva sección
Antes de generar QA, ejecuta este análisis:

In [ ]:
import pandas as pd

rows = []

for item in dataset:

    rows.append({
        "documento": item["documento"],
        "categoria": item["categoria"],
        "caracteres": len(item["texto"]),
        "palabras": len(item["texto"].split())
    })

df = pd.DataFrame(rows)

print(df.describe())

          caracteres      palabras
count      10.000000     10.000000
mean   104201.800000  14360.300000
std    129891.397601  17593.589622
min         0.000000      0.000000
25%     17933.250000   2636.750000
50%     34483.500000   5165.000000
75%    150397.250000  20290.500000
max    385607.000000  52073.000000


También identifica documentos sospechosamente pequeños:

In [ ]:
df[df["palabras"] < 100]

,documento,categoria,caracteres,palabras
9,GacetaUPTA.txt,gacetasOficiales,0,0
